# 3. The HBV model

In this chapter the discharge of the Okavango River at Mohembo will be modelled. First, a model is selected based on similar studies of the river. Using similar studies, a hydrological model that is suitable for the Okavango River can be chosen. After the selection, the model will be calibrated and the result will be compared to the observed discharge data. 

## Model selection
Different hydrological models have been used in earlier research to model the Okavango River. One study in 2006 used the Pitman hydrological model to assess the impact of different climate change scenarios on the river flow (Andersson, et al., 2006). More recent studies use the Soil and Water Assessment Tool (SWAT) to model the Okavango River (Homa, et al., 2025; Negussie, et al., 2026). The eWaterCycle does not contain one these models, so for this study the Hydrologiska Bryåns Vattenbalansavdelning (HBV) model is chosen. The reason for this choice is the fact that the HBV model shares some similarities with the Pitman model. Like the Pitman model and SWAT, the HBV model is semi-distributed. Also, the processes that are used in the different models are similar. In figure 3.1, a schematic representation of the HBV model is given (Melotto, et al., 2026). The HBV model is a conceptual catchment model and it has a relatively simple model structure. Despite the simplicity of the model, the HBV model performed well during several model intercomparisons (Seibert & Bergström, 2022). The components of the HBV model with their description are given in table 3.1 (Melotto, et al., 2026).

<div align="center">
<figure>
  <img src="Figures/HBV_model.png" width="400">
  <figcaption><i>Figure 3.1: Schematic representation HBV (Melotto, et al., 2026)</i></figcaption>
</figure>
</div>

|                        |**Inputs**               |**Description**                                |
|------------------------|-------------------------|-----------------------------------------------|
|**Reservoirs**          |Sp                       |Snow reservoir                                 |
|                        |Si                       |Interception reservoir                         |
|                        |Su                       |Unsaturated reservoir                          |
|                        |Sf                       |Fast reservoir                                 |
|                        |Ss                       |Slow reservoir                                 |
|**Inputs**              |P                        |Precipitation (mm/day)                         |
|                        |Ep                       |Potential evaporation (mm/day)                 |
|                        |T                        |Temperature (degree Celsius)                   |
|**Parameters**          |Imax                     |Interception capacity (mm)                     |
|                        |Ce                       |Soil runoff coefficient (-)                    |
|                        |Sumax                    |Max soil moisture storage (mm)                 |
|                        |Beta                     |Shape parameter for runoff generation (-)      |
|                        |Pmax                     |Percolation threshold (mm/day)                 |
|                        |Tlag                     |Routing lag time (days)                        |
|                        |Kf                       |Fast runoff recession coefficient (1/day)      |
|                        |Ks                       |Slow runoff recession coefficient (1/day)      |
|                        |Fm                       |Snowmelt factor (-)                            |

<div align="center">
<i>Table 3.1: Elaboration different components (Melotto, et al., 2026)</i>
</div>

## 3.2 Calibration
The precipitation, temperature and potential evaporation are the inputs for the HBV model. These inputs are retrieved from the ERA5 dataset. In the code below, the way to download the ERA5 forcing data is shown. To download the forcing data, a shapefile of the catchment area is needed. The shapefile for the catchment area of Mohembo is made with QGIS, using HydroBASINS of Africa (HydroSHEDS, n.d.). “HydroBASINS represents a series of vectorized polygon layers that depict sub-basin boundaries at a global scale” (HydroSHEDS, n.d.). With a picture of the catchment area of Mohembo, the sub-basins are selected in QGIS to make the shapefile (Bartsch, et al., 2006).      

In [1]:
#Loading packages
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

import ewatercycle
import ewatercycle.forcing

In [4]:
#Loading shapefile of the catchment area of Mohembo
shapefile_path =  Path.home() / "BEP-beau/book/thesis_projects/BSc/2026_Q4_BeauBuijtenhuijs_CEG/Data/Shapefile" / "CatchmentArea_4326.shp"

#Defining start and end date for the calibration period
calibration_start_date = "1975-01-01T00:00:00Z"
calibration_end_date = "2010-12-31T00:00:00Z"

#Creating path for ERA5 data
forcing_path_ERA5 = Path.home() / "BEP-beau/book/thesis_projects/BSc/2026_Q4_BeauBuijtenhuijs_CEG/Data/ERA5_Plots/own_shapefile_1"
forcing_path_ERA5.mkdir(exist_ok=True, parents=True)

#Downloading ERA5 data
#ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].generate(
#   dataset="ERA5",
#   start_time=calibration_start_date,
#   end_time=calibration_end_date,
#   shape=shapefile_path,
#   directory=forcing_path_ERA5,
#)

#Loading downloaded ERA5 data
load_location = forcing_path_ERA5 / "work" / "diagnostic" / "script"  
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)

For the optimalization of the parameters, the model is calibrated for 80% of the timeframe of the observed discharge of the Okavango River and validated for the remainder 20% of the timeframe. With ERA5 forcing data, the model is calibrated between 1 January 1975 and 31 December 2010 and validated from 1 January 2011 to 31 December 2020. The model simulations were started on 1 January 1970 to minimize the spin-up period. The spin-up period is the period in which the storage fills up. This process causes the model to be far off in the first couple years of the observed discharge data. By starting from 1970, the spin-up is mostly done by 1975, where the observed data begins. 

To calibrate the parameters of the HBV model, 2000 sets of parameters are generated. Each generated parameter has a value that is in the range of realistic values for the optimized parameter. For all the parameters sets, the HBV model will be used and the output will be compared to the observed discharge of the Okavango River at Mohembo. The parameter set that performs the best will be used as the optimized parameters for the HBV model.

First, the Kling-Gupta Efficiency (KGE) is used to optimize the parameters of the HBV model. The KGE is used to determine the accuracy of hydrological models. Using the correlation coefficient, variability ratio, and bias ratio, the KGE returns a value between minus infinity and one. KGE-values closer to one represent a better fit between the model and the observed data (Thieu, 2021). The KGE value is calculated with the following formula:

$$ \mathrm{KGE} = 1 - \sqrt{(r-1)^2 + (\beta-1)^2 + (\gamma-1)^2} $$
$$ \beta = \text{bias ratio} = \frac{\hat{\mu}_y}{\mu_y} $$
$$ \gamma = \text{variability ratio} = \frac{CV_{\hat{y}}}{CV_y} = \frac{\sigma_{\hat{y}} / \hat{\mu}_y}{\sigma_y / \mu_y} $$

$$ Where: $$
$$ r = \text{correlation coefficient} $$
$$ CV = \text{coefficient of variation} $$
$$ \mu = \text{mean} $$
$$ \sigma = \text{standard deviation} $$
$$ \hat{y} = \text{modelled value} $$
$$ y = \text{observed value} $$

Next, the log-NSE method is performed to calibrate the parameters of the HBV model. The Nash-Sutcliffe coefficient (NSE) is widely used to determine the accuracy of hydrological models. The conventional NSE method gives better results when high values are accurately modelled in contrast to the accuracy of modelled low values. The log-NSE is more sensitive to the accuracy of the modelled low values (Rdocumentation, n.d.). This study focuses on droughts, so it is also important that low flows are well modelled. For this reason, the log-NSE is used to calibrate the parameters. The log-NSE gives values between minus infinity and one, where the value 1 represents a perfect fit. The log-NSE is calculated with the following formula:

$$ \text{log NSE} = 1 - \frac{\sum \left( \log(\mathrm{sim}) - \log(\mathrm{obs}) \right)^2}
{\sum \left( \log(\mathrm{obs}) -\log\left(\mathrm{mean}(\mathrm{obs})\right) \right)^2} $$

$$ Where: $$
$$ sim = \text{modelled value} $$
$$ obs = \text{observed value} $$

Lasty, the Root Mean Square Error (RMSE) is used to calibrate the parameters of the HBV model. The RMSE varies between zero and infinity, where values closer to zero represent a better fit. The RMSE is calculated with the following formula:

$$ RMSE = \sqrt{\frac{1}{n}\sum (y - \hat{y})^2} $$

$$ Where: $$
$$ n = \text{number of data points} $$
$$ \hat{y} = \text{modelled value} $$
$$ y = \text{observed value} $$

The modelled discharge, parameters and values of the three different methods is shown in Appendix A.

The reason the parameters are calibrated with the daily discharge data instead of the yearly volumes is the inaccuracy of the modelled daily discharge. When calibrating with yearly volumes, the peaks of the daily discharge were extremely high while the low flows tended to go to almost zero. This showed modelled yearly volumes that were in line with the observed data, but the modelled daily discharge data was extremely inaccurate.  

To further optimize the parameters, the KGE and log-NSE method are combined. The model with the KGE calibrated parameters shows the best result at modelling the peak, while the modelled low flows do not fit the observed data well. The model with parameters from the calibration using the log-NSE method shows better results at modelling the low flows, while the peaks are far from the observed data. Because the KGE and log-NSE methods have the same scale, weights are applied to the individual methods and then the values are summed. The highest value of the combined method shows the best result of the HBV model. It is chosen to apply a weight of 0.7 to the KGE value and a weight of 0.3 to the log-NSE value. These weights show the best results. A model with a higher weight for the log-NSE performed worse at modelling the peaks, while a higher KGE value tended to model the low flows too high. This study focusses on yearly volumes, that is why both the peak and low flows are important.

The calibrated parameters are shown in table 3.2. The value of the combined method for the calibration period is 0.854 and for the validation period it is 0.875. The value of the combined method for the whole timeframe of the observed data is 0.877. The modelled discharge compared to the observed discharge is shown in figure 3.2.
